# 🎙️ VoiceGetter on Google Colab (free GPU, works from your phone)

1. **Runtime → Change runtime type → T4 GPU**.
2. Run cell 1 (install) and cell 2 (start). Tap ▶ on each cell.
3. Cell 2 prints a link and a password. Open the link on your phone.
4. Keep this tab open while it works. Colab stops after some time without activity.

**Private repo?** Add a secret named `GITHUB_TOKEN` (key icon 🔑 on the left) with a GitHub token that can read the repo.

**YouTube says "Sign in to confirm you're not a bot"?** YouTube often blocks Colab servers. Export your YouTube cookies as `cookies.txt` (see README) and upload the file in cell 2. Or upload the audio file on the web page instead of a link.

In [ ]:
#@title 1. Install (about 3-5 minutes)
REPO = "fletchjobrien/voicegetter"  #@param {type:"string"}
BRANCH = ""  #@param {type:"string"}

import os, subprocess
token = ""
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
if not os.path.isdir("/content/voicegetter"):
    cmd = ["git", "clone", "-q", "--depth", "1"] + (["-b", BRANCH] if BRANCH else []) + [url, "/content/voicegetter"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        err = r.stderr.replace(token, "***") if token else r.stderr
        raise SystemExit("Clone failed. Is GITHUB_TOKEN set for a private repo?\n" + err)
else:
    subprocess.run(["git", "-C", "/content/voicegetter", "pull", "-q"])

!pip install -q -r /content/voicegetter/requirements.txt
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - choose a GPU runtime!")
print("Install done.")

In [ ]:
#@title 2. Start and get the phone link
UPLOAD_COOKIES = False  #@param {type:"boolean"}

import os, re, secrets, subprocess, time
os.chdir("/content/voicegetter")
env = dict(os.environ)
if UPLOAD_COOKIES:
    from google.colab import files
    up = files.upload()
    if up:
        name = next(iter(up))
        os.replace(name, "/content/cookies.txt")
        env["VOICEGETTER_COOKIES"] = "/content/cookies.txt"
elif os.path.exists("/content/cookies.txt"):
    env["VOICEGETTER_COOKIES"] = "/content/cookies.txt"

password = env.get("VOICEGETTER_PASSWORD") or secrets.token_urlsafe(6)
env["VOICEGETTER_PASSWORD"] = password
env["VOICEGETTER_DATA"] = "/content/data"

subprocess.run("pkill -f 'app.main' ; pkill -f cloudflared", shell=True)
server = subprocess.Popen(["python", "-m", "app.main"], env=env,
                          stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
link = None
t0 = time.time()
while time.time() - t0 < 60 and not link:
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        link = m.group(0)
if not link:
    raise SystemExit("Could not start the tunnel. Run this cell again.")
print("Open this on your phone:\n")
print("   ", link)
print("\nPassword:", password)
print("\nKeep this tab open. Server log: /content/server.log")

In [ ]:
#@title 3. (Optional) Show the server log
!tail -n 40 /content/server.log